# Supervised Learning Part 2 — Tutorial

**Duration:** 75–90 minutes  
**Primary outcome:** Train ensemble methods (Random Forest, Gradient Boosting, Voting, Stacking) and a simple neural network on digit recognition data, and understand when each method excels.

---

## What we'll cover

| Section | Topic |
|---------|-------|
| 1 | Setup & Dataset Exploration |
| 2 | Why Ensemble Methods? |
| 3 | Random Forest |
| 4 | Gradient Boosting |
| 5 | HistGradientBoosting (Fast Boosting) |
| 6 | Voting Ensemble |
| 7 | Simple Neural Network |
| 8 | Algorithm Comparison |
| 9 | Error Analysis |
| 10 | Practice Exercises |

We'll use the **digits dataset** — 1,797 hand-written digit images (0–9), each represented as an 8×8 pixel grid (64 features). It's a classic multi-class classification problem that really challenges single models.

## Section 1: Setup & Dataset Exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print('All libraries loaded successfully!')

In [ ]:
# Load the digits dataset
digits = load_digits()

X = digits.data       # shape: (1797, 64)
y = digits.target     # shape: (1797,)

print(f'Samples  : {X.shape[0]}')
print(f'Features : {X.shape[1]}  (8x8 pixel grid = 64 pixels per image)')
print(f'Classes  : {len(np.unique(y))}  (digits 0–9)')
print(f'Class labels: {np.unique(y)}')
print()
print('Samples per class:')
for cls in np.unique(y):
    print(f'  Digit {cls}: {np.sum(y == cls)} samples')

In [ ]:
# Visualise 20 sample digit images in a 4x5 grid
fig, axes = plt.subplots(4, 5, figsize=(10, 8))
fig.suptitle('Sample Digit Images from the Dataset', fontsize=14, fontweight='bold')

sample_indices = np.random.choice(len(X), 20, replace=False)

for ax, idx in zip(axes.ravel(), sample_indices):
    ax.imshow(digits.images[idx], cmap='gray_r', interpolation='nearest')
    ax.set_title(f'Label: {y[idx]}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('Each image is stored as a flattened array of 64 pixel values (0–16 grayscale).')

In [ ]:
# Train/test split — stratified so every digit is proportionally represented
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features — important for SVC and MLP
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')
print(f'Split ratio  : {X_train.shape[0]/len(X):.0%} / {X_test.shape[0]/len(X):.0%}')

In [ ]:
# Helper function — reuse throughout the notebook
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, scaled=False):
    """Fit model, print metrics, return (train_acc, test_acc, fit_time)."""
    start = time.time()
    model.fit(X_tr, y_tr)
    fit_time = time.time() - start

    train_acc = model.score(X_tr, y_tr)
    test_acc  = model.score(X_te, y_te)

    print(f'=== {name} ===')
    print(f'  Fit time  : {fit_time:.2f}s')
    print(f'  Train acc : {train_acc:.4f}')
    print(f'  Test acc  : {test_acc:.4f}')
    print()
    print(classification_report(y_te, model.predict(X_te), target_names=[str(i) for i in range(10)]))
    return train_acc, test_acc, fit_time

print('evaluate_model() helper ready.')

## Section 2: Why Ensemble Methods?

### The "Wisdom of Crowds" Analogy

Imagine you're trying to guess the number of jelly beans in a jar. If you ask one person, they might be way off. But if you ask 500 people and average their guesses, you'll usually land very close to the truth. The errors of individuals cancel each other out.

Machine learning ensembles work the same way: combine many imperfect models so their individual mistakes cancel each other out.

---

### Three Families of Ensembles

```
BAGGING (parallel)               BOOSTING (sequential)           STACKING
─────────────────────            ─────────────────────────       ─────────────────────
  Data                           Data                            Data
   │                              │                               │
   ├── Bootstrap sample 1        Model 1 → errors                ├── Base model 1 ──┐
   ├── Bootstrap sample 2        Model 2 corrects errors         ├── Base model 2 ──┤→ Meta-model → ŷ
   └── Bootstrap sample N        Model 3 corrects residuals      └── Base model 3 ──┘
         ↓                              ↓
   Model 1, Model 2 … N          Final = weighted sum
         ↓
   Vote / Average → ŷ
```

| Method | Parallel? | Reduces | Example |
|--------|-----------|---------|--------|
| Bagging | Yes | Variance | Random Forest |
| Boosting | No (sequential) | Bias | Gradient Boosting, XGBoost |
| Stacking | Depends | Both | StackingClassifier |

Let's verify the intuition: can three mediocre classifiers outperform any single one?

In [ ]:
from sklearn.ensemble import VotingClassifier

# Three shallow decision trees — each intentionally weak (max_depth=3)
dt1 = DecisionTreeClassifier(max_depth=3, random_state=1)
dt2 = DecisionTreeClassifier(max_depth=3, random_state=2)
dt3 = DecisionTreeClassifier(max_depth=3, random_state=3)

voting = VotingClassifier(
    estimators=[('dt1', dt1), ('dt2', dt2), ('dt3', dt3)],
    voting='hard'
)

results = {}
for name, clf in [('DT-1 (depth=3)', dt1), ('DT-2 (depth=3)', dt2),
                  ('DT-3 (depth=3)', dt3), ('Voting (3 DTs)', voting)]:
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    results[name] = acc
    print(f'{name:25s}  test acc = {acc:.4f}')

print()
best_single = max(v for k, v in results.items() if 'Voting' not in k)
ensemble_acc = results['Voting (3 DTs)']
print(f'Best single tree : {best_single:.4f}')
print(f'Voting ensemble  : {ensemble_acc:.4f}')
print(f'Improvement      : +{ensemble_acc - best_single:.4f}')

## Section 3: Random Forest

### How it works

Random Forest = **Bagging + Feature Randomness**

1. Draw a bootstrap sample from the training data (random rows, with replacement).
2. At each node split, consider only a **random subset of features** (typically √n_features).
3. Grow a full decision tree on each sample.
4. Aggregate predictions by majority vote.

The feature randomness is the key innovation over plain bagging — it **de-correlates** the trees so their errors don't overlap.

**Out-of-Bag (OOB) score:** Because each tree is trained on ~63% of the data (bootstrap), the remaining ~37% can be used as a free validation set. OOB score is a built-in cross-validation estimate.

In [ ]:
# Fit Random Forest and evaluate
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1)

rf_train_acc, rf_test_acc, rf_fit_time = evaluate_model(
    'Random Forest (100 trees)',
    rf, X_train, y_train, X_test, y_test
)

print(f'OOB score (built-in validation): {rf.oob_score_:.4f}')
print('OOB score is a free estimate of generalisation — no separate val set needed.')

In [ ]:
# Feature importance heatmap — which pixels matter most?
importances = rf.feature_importances_.reshape(8, 8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: heatmap of importances on the 8x8 grid
im = axes[0].imshow(importances, cmap='hot', interpolation='nearest')
axes[0].set_title('RF Feature Importance\n(Which pixels matter?)', fontsize=12)
axes[0].set_xticks(range(8))
axes[0].set_yticks(range(8))
axes[0].set_xlabel('Pixel column')
axes[0].set_ylabel('Pixel row')
plt.colorbar(im, ax=axes[0], label='Importance')

# Right: ranked bar chart of top-20 pixel importances
top20 = np.argsort(rf.feature_importances_)[::-1][:20]
axes[1].bar(range(20), rf.feature_importances_[top20], color='steelblue')
axes[1].set_xticks(range(20))
axes[1].set_xticklabels([f'px{i}' for i in top20], rotation=45, ha='right', fontsize=8)
axes[1].set_title('Top 20 Most Important Pixels', fontsize=12)
axes[1].set_ylabel('Importance score')

plt.tight_layout()
plt.show()

print('Bright pixels = high importance. Centre pixels tend to matter more than edges!')

In [ ]:
# Effect of n_estimators on test accuracy
n_trees_list = [10, 50, 100, 200]
rf_n_tree_accs = []

for n in n_trees_list:
    clf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    rf_n_tree_accs.append(clf.score(X_test, y_test))
    print(f'  n_estimators={n:4d}  →  test acc = {rf_n_tree_accs[-1]:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(n_trees_list, rf_n_tree_accs, marker='o', linewidth=2, color='steelblue')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Test Accuracy')
plt.title('Random Forest: Accuracy vs Number of Trees')
plt.xticks(n_trees_list)
plt.ylim(0.94, 1.0)
plt.tight_layout()
plt.show()

print()
print('Accuracy improves quickly with more trees, then plateaus. Diminishing returns after ~100.')

## Section 4: Gradient Boosting

### How it works

Gradient Boosting builds trees **sequentially**. Each new tree focuses on the **residual errors** (gradients of the loss) left by all previous trees.

```
Iteration 1:  Tree₁ → prediction₁  (rough estimate)
Iteration 2:  Tree₂ trained on residuals of Tree₁
Iteration 3:  Tree₃ trained on residuals of Tree₁ + Tree₂
      ⋮
Final:  ŷ = lr * (Tree₁ + Tree₂ + … + TreeN)
```

The **learning rate** controls how much each new tree contributes. A lower learning rate requires more trees but generalises better.

### Boosting vs Bagging

| | Bagging (RF) | Boosting (GB) |
|--|--|--|
| Tree order | Parallel | Sequential |
| Reduces | Variance | Bias |
| Risk | — | Overfitting if LR too high |
| Speed | Fast (parallel) | Slower (sequential) |
| Tuning | Few hyperparams | More hyperparams |

In [ ]:
# Fit Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

gb_train_acc, gb_test_acc, gb_fit_time = evaluate_model(
    'Gradient Boosting (100 estimators, lr=0.1)',
    gb, X_train, y_train, X_test, y_test
)

In [ ]:
# Training loss vs n_estimators — observe convergence
# Use staged_predict to get cumulative loss at each stage
gb_staged_accs = []
for y_pred_staged in gb.staged_predict(X_test):
    gb_staged_accs.append(np.mean(y_pred_staged == y_test))

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(gb_staged_accs) + 1), gb_staged_accs,
         color='darkorange', linewidth=2, label='GB test accuracy')
plt.axhline(rf_test_acc, color='steelblue', linestyle='--',
            linewidth=1.5, label=f'Random Forest ({rf_test_acc:.4f})')
plt.xlabel('Number of boosting iterations')
plt.ylabel('Test Accuracy')
plt.title('Gradient Boosting: Accuracy vs Boosting Rounds')
plt.legend()
plt.tight_layout()
plt.show()

print(f'RF  test acc  : {rf_test_acc:.4f}')
print(f'GB  test acc  : {gb_test_acc:.4f}')

## Section 5: HistGradientBoosting (Fast Boosting)

Scikit-learn's `HistGradientBoostingClassifier` is inspired by **LightGBM** / **XGBoost**. Instead of trying every possible split value, it **bins continuous features into histograms** (256 bins by default) before finding splits.

This makes it dramatically faster on large datasets while achieving similar — or better — accuracy.

### When to use which?

| Situation | Use |
|-----------|-----|
| Small dataset (< 10k samples), interpretability needed | `GradientBoostingClassifier` |
| Large dataset, need speed | `HistGradientBoostingClassifier` |
| Production with tabular data, maximum accuracy | `HistGB` or XGBoost/LightGBM |
| Missing values in data | `HistGB` handles them natively |


In [ ]:
# Fit HistGradientBoosting and compare speed + accuracy
hgb = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, random_state=42)

hgb_train_acc, hgb_test_acc, hgb_fit_time = evaluate_model(
    'HistGradientBoosting (100 iters, lr=0.1)',
    hgb, X_train, y_train, X_test, y_test
)

# Side-by-side speed comparison
print('\n--- Speed & Accuracy Comparison ---')
print(f'{"Model":<35} {"Test Acc":>10} {"Fit Time":>10}')
print('-' * 58)
print(f'{"GradientBoosting":<35} {gb_test_acc:>10.4f} {gb_fit_time:>9.2f}s')
print(f'{"HistGradientBoosting":<35} {hgb_test_acc:>10.4f} {hgb_fit_time:>9.2f}s')
print()
if hgb_fit_time < gb_fit_time:
    print(f'HistGB was {gb_fit_time / hgb_fit_time:.1f}x faster than plain GradientBoosting.')
else:
    print('On this small dataset the speed difference may be minimal.')

## Section 6: Voting Ensemble

### Soft vs Hard Voting

- **Hard voting**: each model votes for a class label; majority wins. Simple but ignores confidence.
- **Soft voting**: average the predicted **probabilities** across models, then pick the highest. More nuanced — a highly confident model has more influence.

```
Example: predict digit '3'

Hard voting:                    Soft voting:
RF  → 3   ✓                    RF  → P(3) = 0.85
GB  → 3   ✓                    GB  → P(3) = 0.72
SVC → 5   ✗                    SVC → P(3) = 0.10
                                Avg P(3) = 0.557  ← still wins
Vote = 3  (2/3)                 Vote = 3
```

Soft voting generally performs better when all base models support `predict_proba`.

In [ ]:
# Voting ensemble: RF + GB + SVC (with probability=True)
# Use scaled data for SVC
rf_v  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
gb_v  = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
svc_v = SVC(kernel='rbf', probability=True, random_state=42)

voting_clf = VotingClassifier(
    estimators=[('rf', rf_v), ('gb', gb_v), ('svc', svc_v)],
    voting='soft'
)

voting_train_acc, voting_test_acc, voting_fit_time = evaluate_model(
    'Voting Ensemble (RF + GB + SVC, soft)',
    voting_clf, X_train_sc, y_train, X_test_sc, y_test
)

In [ ]:
# Compare individual components vs voting ensemble
component_results = {}

for name, clf in [('RF', rf_v), ('GB', gb_v), ('SVC', svc_v)]:
    acc = clf.score(X_test_sc, y_test)   # already fitted inside VotingClassifier
    component_results[name] = acc
    print(f'{name:6s} test acc = {acc:.4f}')

component_results['Voting'] = voting_test_acc
print(f'{"Voting":6s} test acc = {voting_test_acc:.4f}')

plt.figure(figsize=(7, 4))
colors = ['steelblue', 'steelblue', 'steelblue', 'darkorange']
bars = plt.bar(component_results.keys(), component_results.values(), color=colors)
plt.ylabel('Test Accuracy')
plt.title('Individual Models vs Voting Ensemble')
plt.ylim(0.95, 1.0)
for bar, val in zip(bars, component_results.values()):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.0002,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## Section 7: Simple Neural Network

### MLPClassifier — Multi-Layer Perceptron

A neural network with one or more hidden layers. Each layer applies:

```
output = activation(W · input + b)
```

We use **ReLU** (Rectified Linear Unit) activation: `f(x) = max(0, x)`. It's fast, avoids vanishing gradients, and works well for most problems.

Our architecture:
```
Input (64 pixels) → Hidden Layer 1 (128 neurons, ReLU)
                  → Hidden Layer 2 (64 neurons, ReLU)
                  → Output (10 classes, Softmax)
```

### When do NNs outperform ensembles?

- **Very large datasets** (millions of samples) — NNs scale better
- **Unstructured data** (images, text, audio) — NNs can learn spatial/sequential patterns
- **Transfer learning scenarios** — pretrained NNs can be fine-tuned

For **tabular data** like digits' flat pixel vectors, ensembles often win or tie with much less tuning.

In [ ]:
# Fit MLP on scaled data
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=200,
    random_state=42
)

mlp_train_acc, mlp_test_acc, mlp_fit_time = evaluate_model(
    'MLP Neural Network (128→64, ReLU)',
    mlp, X_train_sc, y_train, X_test_sc, y_test
)

In [ ]:
# Plot the training loss curve
plt.figure(figsize=(9, 4))
plt.plot(mlp.loss_curve_, color='crimson', linewidth=2)
plt.xlabel('Training Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('MLP Neural Network: Training Loss Curve')
plt.tight_layout()
plt.show()

print(f'Converged in {len(mlp.loss_curve_)} epochs.')
print(f'Final training loss: {mlp.loss_curve_[-1]:.6f}')

In [ ]:
# Confusion matrix for the MLP
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_estimator(
    mlp, X_test_sc, y_test,
    display_labels=list(range(10)),
    cmap='Blues',
    ax=ax
)
ax.set_title('MLP Confusion Matrix (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

print('Rows = true label, Columns = predicted label.')
print('Off-diagonal cells are misclassifications.')

## Section 8: Algorithm Comparison

Let's put all five algorithms head-to-head using the same train/test split.

In [ ]:
# Collect results — refit each on original (unscaled) where applicable
# RF and GB do not need scaling; SVC/MLP do.

comparison_data = [
    {
        'Algorithm'    : 'Random Forest',
        'Train Acc'    : rf_train_acc,
        'Test Acc'     : rf_test_acc,
        'Fit Time (s)' : rf_fit_time,
    },
    {
        'Algorithm'    : 'Gradient Boosting',
        'Train Acc'    : gb_train_acc,
        'Test Acc'     : gb_test_acc,
        'Fit Time (s)' : gb_fit_time,
    },
    {
        'Algorithm'    : 'HistGradientBoosting',
        'Train Acc'    : hgb_train_acc,
        'Test Acc'     : hgb_test_acc,
        'Fit Time (s)' : hgb_fit_time,
    },
    {
        'Algorithm'    : 'Voting Ensemble',
        'Train Acc'    : voting_train_acc,
        'Test Acc'     : voting_test_acc,
        'Fit Time (s)' : voting_fit_time,
    },
    {
        'Algorithm'    : 'MLP Neural Network',
        'Train Acc'    : mlp_train_acc,
        'Test Acc'     : mlp_test_acc,
        'Fit Time (s)' : mlp_fit_time,
    },
]

df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.sort_values('Test Acc', ascending=False).reset_index(drop=True)
df_comparison['Train Acc']    = df_comparison['Train Acc'].round(4)
df_comparison['Test Acc']     = df_comparison['Test Acc'].round(4)
df_comparison['Fit Time (s)'] = df_comparison['Fit Time (s)'].round(2)

print(df_comparison.to_string(index=False))

In [ ]:
# Sorted bar chart of test accuracies
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Test accuracy
axes[0].barh(
    df_comparison['Algorithm'],
    df_comparison['Test Acc'],
    color=['gold' if i == 0 else 'steelblue' for i in range(len(df_comparison))]
)
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Test Accuracy by Algorithm')
axes[0].set_xlim(0.95, 1.0)
for i, (acc, alg) in enumerate(zip(df_comparison['Test Acc'], df_comparison['Algorithm'])):
    axes[0].text(acc + 0.0002, i, f'{acc:.4f}', va='center', fontsize=9)

# Fit time
axes[1].barh(
    df_comparison['Algorithm'],
    df_comparison['Fit Time (s)'],
    color='darkorange'
)
axes[1].set_xlabel('Fit Time (seconds)')
axes[1].set_title('Training Time by Algorithm')
for i, t in enumerate(df_comparison['Fit Time (s)']):
    axes[1].text(t + 0.05, i, f'{t:.1f}s', va='center', fontsize=9)

plt.suptitle('Algorithm Comparison on Digits Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Pros, Cons, and When to Use Each

| Algorithm | Pros | Cons | Best When |
|-----------|------|------|-----------|
| **Random Forest** | Fast, robust, OOB score, feature importance | Not great with very sparse/high-dim data | Medium datasets, interpretability needed |
| **Gradient Boosting** | Often highest accuracy on tabular data | Slow to train, many hyperparams | Tabular, competitive ML |
| **HistGradientBoosting** | Fast, handles missing values natively | Slightly less interpretable than plain GB | Large tabular datasets |
| **Voting Ensemble** | Reduces variance by combining diverse models | Slow (trains all base models) | When you have diverse, well-tuned base models |
| **MLP Neural Network** | Flexible, learns representations | Needs scaling, many hyperparams, slower to tune | Large data, complex feature interactions |

## Section 9: Error Analysis

Knowing *that* a model makes mistakes is not enough — we need to know *which* mistakes it makes and *why*.

We'll use the best-performing model from our comparison.

In [ ]:
# Pick the best model by test accuracy
best_name = df_comparison.iloc[0]['Algorithm']
print(f'Best model: {best_name}  (test acc = {df_comparison.iloc[0]["Test Acc"]:.4f})')

# Get the fitted model object
model_lookup = {
    'Random Forest'        : (rf,           X_test),
    'Gradient Boosting'    : (gb,           X_test),
    'HistGradientBoosting' : (hgb,          X_test),
    'Voting Ensemble'      : (voting_clf,   X_test_sc),
    'MLP Neural Network'   : (mlp,          X_test_sc),
}

best_model, best_X_test = model_lookup[best_name]
y_pred_best = best_model.predict(best_X_test)

# Full confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=list(range(10)),
    cmap='Blues',
    ax=ax
)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Find misclassified examples
misclassified_idx = np.where(y_pred_best != y_test)[0]
print(f'Total misclassified: {len(misclassified_idx)} out of {len(y_test)} test samples')
print()

# Show which pairs of digits are most confused
from collections import Counter
confusion_pairs = Counter()
for idx in misclassified_idx:
    confusion_pairs[(y_test[idx], y_pred_best[idx])] += 1

print('Most common confusion pairs (true → predicted):')
for (true, pred), count in confusion_pairs.most_common(10):
    print(f'  {true} → {pred} : {count} times')

In [ ]:
# Visualise up to 9 misclassified images
n_show = min(9, len(misclassified_idx))
sample_errors = misclassified_idx[:n_show]

fig, axes = plt.subplots(3, 3, figsize=(7, 7))
fig.suptitle(f'Misclassified Digits — {best_name}', fontsize=13, fontweight='bold')

for ax, idx in zip(axes.ravel(), sample_errors):
    # X_test maps back to the original image via the test indices
    img = X_test[idx].reshape(8, 8)
    ax.imshow(img, cmap='gray_r', interpolation='nearest')
    ax.set_title(f'True: {y_test[idx]}  Pred: {y_pred_best[idx]}',
                 fontsize=9, color='red')
    ax.axis('off')

# Hide unused subplots if fewer than 9 errors
for ax in axes.ravel()[n_show:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

print('Notice: many errors occur on digits that look visually similar (e.g., 3 vs 8, 4 vs 9).')

## Section 10: Practice Exercises

Work through these exercises to deepen your understanding. Collapsed solution cells follow each exercise.

---

### Exercise 1

Tune `n_estimators` for Random Forest across values `[10, 50, 100, 200, 500]`. Plot test accuracy vs n_estimators. At which point do you see diminishing returns? Also record and plot fit time.

In [ ]:
# Exercise 1 — your code here
# Hint: loop over [10, 50, 100, 200, 500], fit a RandomForestClassifier,
# record (test_acc, fit_time) for each, then plot both.


In [ ]:
# Solution — Exercise 1
n_list = [10, 50, 100, 200, 500]
ex1_accs, ex1_times = [], []

for n in n_list:
    t0 = time.time()
    clf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    ex1_times.append(time.time() - t0)
    ex1_accs.append(clf.score(X_test, y_test))
    print(f'n={n:4d}  acc={ex1_accs[-1]:.4f}  time={ex1_times[-1]:.2f}s')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(n_list, ex1_accs, marker='o', color='steelblue')
ax1.set_xlabel('n_estimators')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('RF Accuracy vs n_estimators')

ax2.plot(n_list, ex1_times, marker='s', color='darkorange')
ax2.set_xlabel('n_estimators')
ax2.set_ylabel('Fit Time (s)')
ax2.set_title('RF Fit Time vs n_estimators')

plt.tight_layout()
plt.show()

---

### Exercise 2

Experiment with `MLPClassifier` hidden layer architectures. Try at least four configurations:
- `(64,)` — one small hidden layer
- `(128, 64)` — our baseline
- `(256, 128, 64)` — three layers
- `(512,)` — one wide layer

Plot test accuracy for each. Which architecture gives the best accuracy?

In [ ]:
# Exercise 2 — your code here
# Hint: use X_train_sc / X_test_sc (scaled data), loop over architectures,
# fit each MLPClassifier, record test_acc.


In [ ]:
# Solution — Exercise 2
architectures = [(64,), (128, 64), (256, 128, 64), (512,)]
arch_labels   = ['(64,)', '(128,64)', '(256,128,64)', '(512,)']
ex2_accs = []

for arch, label in zip(architectures, arch_labels):
    clf = MLPClassifier(hidden_layer_sizes=arch, activation='relu',
                        max_iter=200, random_state=42)
    clf.fit(X_train_sc, y_train)
    acc = clf.score(X_test_sc, y_test)
    ex2_accs.append(acc)
    print(f'{label:20s}  test acc = {acc:.4f}')

plt.figure(figsize=(8, 4))
plt.bar(arch_labels, ex2_accs, color='crimson')
plt.ylabel('Test Accuracy')
plt.title('MLP Architecture Comparison')
plt.ylim(0.95, 1.0)
for i, acc in enumerate(ex2_accs):
    plt.text(i, acc + 0.001, f'{acc:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---

### Exercise 3

Build a **Stacking Classifier** using:
- Base estimators: `RandomForestClassifier`, `GradientBoostingClassifier`, `MLPClassifier`
- Meta-estimator: `LogisticRegression`

Train on the scaled data and compare its test accuracy to the voting ensemble. Does stacking outperform voting?

> **Tip:** `StackingClassifier` from `sklearn.ensemble` uses cross-validation internally to generate out-of-fold predictions for the meta-learner — it won't overfit on the training labels.

In [ ]:
# Exercise 3 — your code here
# from sklearn.ensemble import StackingClassifier
# from sklearn.linear_model import LogisticRegression


In [ ]:
# Solution — Exercise 3
from sklearn.linear_model import LogisticRegression

base_estimators = [
    ('rf',  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('gb',  GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)),
    ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu',
                          max_iter=200, random_state=42)),
]

stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,
    n_jobs=-1
)

stk_train_acc, stk_test_acc, stk_fit_time = evaluate_model(
    'Stacking Ensemble (RF + GB + MLP → LR)',
    stacking_clf, X_train_sc, y_train, X_test_sc, y_test
)

print(f'Voting Ensemble test acc  : {voting_test_acc:.4f}')
print(f'Stacking Ensemble test acc: {stk_test_acc:.4f}')
delta = stk_test_acc - voting_test_acc
direction = 'better' if delta > 0 else 'worse' if delta < 0 else 'equal'
print(f'Stacking is {direction} by {abs(delta):.4f}.')

---

## Summary

In this tutorial you:

1. **Explored** the digits dataset — 1,797 samples, 64 features, 10 classes.
2. **Understood** why ensembles beat single models through the wisdom-of-crowds effect.
3. **Trained a Random Forest** and inspected which pixels it finds important.
4. **Trained Gradient Boosting** and watched accuracy grow with boosting rounds.
5. **Compared HistGradientBoosting** — same accuracy, much faster.
6. **Built a Voting Ensemble** combining RF + GB + SVC with soft voting.
7. **Trained a Neural Network** (MLP) and examined its loss curve and confusion matrix.
8. **Compared all five algorithms** in a single table and chart.
9. **Analysed errors** — which digits get confused and why.
10. **Practiced** tuning n_estimators, MLP architectures, and building a Stacking Classifier.

### Key Takeaways

- Ensemble methods almost always outperform single models on tabular data.
- **Random Forest** is a great default: fast, robust, interpretable via feature importance.
- **Gradient Boosting** is often the top performer on tabular tasks but requires more tuning.
- **HistGradientBoosting** offers GB-level accuracy with much better scalability.
- **Voting/Stacking** adds another layer — combine diverse, well-tuned models.
- **Neural Networks** shine on large, unstructured data but are overkill for small tabular tasks.
- Always do **error analysis** — it reveals the next lever to pull.